In [1]:
# ============================================================
# BBO NEXT SUBMISSION NOTEBOOK
# Transparency + Interpretability Optimisation Strategy
# Uses Week 1–9 inputs and outputs directly
# ============================================================

import numpy as np
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# WEEK 1–9 INPUTS
# ============================================================

inputs_by_week = [
    [
        np.array([0.211325, 0.788675]),
        np.array([0.723607, 0.276393]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.125000, 0.375000, 0.625000, 0.875000]),
        np.array([0.875000, 0.625000, 0.375000, 0.125000]),
        np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
        np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
        np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
    ],
    [
        np.array([0.654299, 0.353479]),
        np.array([0.754299, 0.253479]),
        np.array([0.304299, 0.553479, 0.709691]),
        np.array([0.804299, 0.603479, 0.409691, 0.205016]),
        np.array([0.854299, 0.653479, 0.359691, 0.155016]),
        np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
        np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
        np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.712865, 0.284413]),
        np.array([0.118496, 0.481282, 0.876608]),
        np.array([1.000000, 0.683447, 0.334333, 0.000000]),
        np.array([0.882245, 0.615032, 0.380358, 0.114494]),
        np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
        np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
        np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
    ],
    [
        np.array([0.183704, 0.127166]),
        np.array([0.255353, 0.749841]),
        np.array([0.094906, 0.770362, 0.859589]),
        np.array([0.820856, 0.226975, 0.784625, 0.113609]),
        np.array([0.917860, 0.273128, 0.475575, 0.052446]),
        np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
        np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
        np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.684812, 0.281383]),
        np.array([0.171685, 0.508728, 0.819757]),
        np.array([0.734042, 0.687625, 0.466427, 0.303264]),
        np.array([0.568707, 0.909615, 0.957143, 0.011420]),
        np.array([0.744968, 0.084644, 0.801527, 0.169728, 0.990511]),
        np.array([1.000000, 0.229371, 0.690054, 0.244152, 0.521424, 1.000000]),
        np.array([0.022620, 0.000000, 0.288709, 0.762817, 0.551656, 0.961269, 0.843087, 1.000000])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.996744, 0.999561]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.743137, 0.718436, 0.581810, 0.365984]),
        np.array([0.572707, 0.914615, 0.952143, 0.016420]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.024461, 0.059677, 0.995330, 0.299568, 0.038576, 0.691027]),
        np.array([0.148189, 0.189483, 0.270757, 0.392507, 0.446351, 0.670593, 0.665417, 1.000000])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.744767, 0.322712]),
        np.array([0.167431, 0.503246, 0.833129]),
        np.array([0.692844, 0.522126, 0.400511, 0.337375]),
        np.array([0.543363, 0.994173, 0.995000, 0.003824]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.194327, 0.008117, 0.940397, 0.331469, 0.120428, 0.708672]),
        np.array([0.208946, 0.148993, 0.202639, 0.316908, 0.431312, 0.503474, 0.634079, 0.995000])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.723607, 0.276393]),
        np.array([0.173580, 0.509985, 0.861627]),
        np.array([0.605953, 0.443908, 0.321881, 0.510101]),
        np.array([0.503541, 0.995000, 0.995000, 0.000001]),
        np.array([0.339419, 0.513193, 0.599128, 0.825009, 0.789136]),
        np.array([0.232309, 0.000001, 0.881882, 0.376082, 0.191527, 0.731904]),
        np.array([0.045850, 0.409304, 0.057579, 0.179582, 0.499647, 0.251108, 0.757574, 0.939082])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.694340, 0.207302]),
        np.array([0.147117, 0.526235, 0.849011]),
        np.array([0.524137, 0.403433, 0.220030, 0.482874]),
        np.array([0.577252, 0.995000, 0.995000, 0.000001]),
        np.array([0.482662, 0.445135, 0.505979, 0.901513, 0.790255]),
        np.array([0.234062, 0.000001, 0.862390, 0.432759, 0.245568, 0.773410]),
        np.array([0.246667, 0.211931, 0.156108, 0.312021, 0.424946, 0.497662, 0.627877, 0.995000])
    ]
]

# ============================================================
# WEEK 1–9 OUTPUTS
# ============================================================

outputs_by_week = [
    np.array([1.1327056288856873e-125, 0.5675786892822564, -0.032385408076210126, -17.20744048260943, 60.223125, -1.3287857969718009, 0.34356041660314957, 9.0501517903694]),
    np.array([1.1867858443665185e-41, 0.2715258567299176, -0.1198581070659559, -13.082213230390916, 50.179390256321376, -1.5080002951000964, 0.31639485635485903, 9.0219949134074]),
    np.array([7.99676981308551e-19, 0.5213723465552891, -0.04726977098841539, -25.67625344929702, 64.78245026474816, -1.7372122723701597, 0.3507828450585503, 9.0587238074074]),
    np.array([-2.410121917336285e-100, 0.0244939290195959, -0.04207093453964322, -20.330739763644917, 61.278397876784794, -2.4624737429462145, 0.0634803557047261, 8.275283250689899]),
    np.array([7.99676981308551e-19, 0.4258127251524913, -0.06232295859499999, -12.496845976106417, 942.2521944777988, -2.280900502246122, 0.21828066675598462, 8.427686115001]),
    np.array([7.99676981308551e-19, -0.004122974640885927, -0.032385408076210126, -14.192389833871584, 943.6841142765069, -1.2257941580841207, 0.4410410448155631, 9.3346877420455]),
    np.array([7.99676981308551e-19, 0.3799297936947829, -0.03240117791304375, -6.92709780967855, 1723.425126888365, -1.2923495282910902, 0.919847131115406, 9.472143824862]),
    np.array([7.99676981308551e-19, 0.5381696913344642, -0.02792717356049581, -4.634066674167709, 1679.193994434957, -1.0832489053663208, 1.3511467710792968, 9.1699591109441]),
    np.array([7.99676981308551e-19, 0.5123530018001279, -0.025522227618805376, -4.5906554634204895, 1784.4636825839586, -1.153199093198391, 1.3736252613501883, 9.472748274068])
]

# ============================================================
# FUNCTIONS
# ============================================================

def get_history(fn_idx):
    X = np.vstack([week[fn_idx] for week in inputs_by_week])
    y = np.array([week[fn_idx] for week in outputs_by_week])
    return X, y

def scaled_target(y):
    return np.sign(y) * np.log1p(np.abs(y))

def train_models(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    ys = scaled_target(y)

    gp = GaussianProcessRegressor(
        kernel=ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1e-5),
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )
    gp.fit(Xs, ys)

    rf = RandomForestRegressor(
        n_estimators=400,
        max_depth=4,
        random_state=42
    )
    rf.fit(X, ys)

    return gp, rf, scaler

def feature_importance_report(rf, X, y):
    importances = rf.feature_importances_
    return importances / (importances.sum() + 1e-9)

def local_sensitivity(model, scaler, x, fn_idx):
    eps = 0.01
    base = model.predict(scaler.transform(x.reshape(1, -1)))[0]
    sens = []

    for i in range(len(x)):
        xp = x.copy()
        xp[i] = np.clip(xp[i] + eps, 0.000001, 0.995000)
        pred = model.predict(scaler.transform(xp.reshape(1, -1)))[0]
        sens.append(pred - base)

    return np.array(sens)

def generate_candidates(X, y, fn_idx):
    dim = X.shape[1]
    order = np.argsort(y)[::-1]
    best_x = X[order[0]]
    second_x = X[order[1]]

    # Functions 3, 4, 6 are still negative: use wider interpretable exploration.
    if fn_idx in [2, 3, 5]:
        local_radius = 0.075
        second_radius = 0.100
        global_n = 3500

    # Function 5 is strong: exploit tightly.
    elif fn_idx == 4:
        local_radius = 0.008
        second_radius = 0.015
        global_n = 300

    # Functions 7 and 8 are strong: moderate exploitation.
    elif fn_idx in [6, 7]:
        local_radius = 0.025
        second_radius = 0.045
        global_n = 1000

    else:
        local_radius = 0.050
        second_radius = 0.075
        global_n = 2000

    local_best = np.clip(best_x + np.random.normal(0, local_radius, size=(12000, dim)), 0.000001, 0.995000)
    local_second = np.clip(second_x + np.random.normal(0, second_radius, size=(6000, dim)), 0.000001, 0.995000)

    mix = np.random.beta(2, 1, size=(2500, 1))
    interp = np.clip(
        mix * best_x + (1 - mix) * second_x + np.random.normal(0, 0.020, size=(2500, dim)),
        0.000001,
        0.995000
    )

    global_search = np.random.uniform(0.000001, 0.995000, size=(global_n, dim))

    return np.vstack([local_best, local_second, interp, global_search, X]), best_x

def propose_next(fn_idx):
    X, y = get_history(fn_idx)
    gp, rf, scaler = train_models(X, y)

    candidates, best_x = generate_candidates(X, y, fn_idx)
    candidates_scaled = scaler.transform(candidates)

    gp_mu, gp_sigma = gp.predict(candidates_scaled, return_std=True)
    rf_pred = rf.predict(candidates)

    # Interpretable ensemble prediction
    mu = 0.65 * gp_mu + 0.35 * rf_pred

    distance = np.linalg.norm(candidates - best_x, axis=1)

    if fn_idx in [2, 3, 5]:
        beta = 1.50
        penalty = 0.035
        strategy = "Wider exploration because output is still negative."
    elif fn_idx == 4:
        beta = 0.10
        penalty = 0.25
        strategy = "Tight exploitation because Function 5 is highly positive."
    elif fn_idx in [6, 7]:
        beta = 0.55
        penalty = 0.08
        strategy = "Moderate exploitation because performance is improving."
    else:
        beta = 1.00
        penalty = 0.05
        strategy = "Balanced search."

    score = mu + beta * gp_sigma - penalty * distance
    selected = candidates[np.argmax(score)]

    importance = feature_importance_report(rf, X, scaled_target(y))
    sensitivity = local_sensitivity(gp, scaler, best_x, fn_idx)

    return np.round(selected, 6), np.max(y), best_x, importance, sensitivity, strategy

# ============================================================
# GENERATE NEXT SUBMISSION
# ============================================================

next_inputs = []

print("INTERPRETABLE BBO NEXT QUERY GENERATION")
print("======================================\n")

for fn_idx in range(8):
    x_next, best_y, best_x, importance, sensitivity, strategy = propose_next(fn_idx)
    next_inputs.append(x_next)

    print(f"Function {fn_idx + 1}")
    print(f"Best output so far: {best_y}")
    print(f"Best input so far:  {np.round(best_x, 6)}")
    print(f"Strategy:           {strategy}")
    print(f"Feature importance: {np.round(importance, 4)}")
    print(f"Local sensitivity:  {np.round(sensitivity, 4)}")
    print(f"Next input:         {x_next}")
    print()

print("PORTAL-READY INPUTS")
print("===================\n")

for i, arr in enumerate(next_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

INTERPRETABLE BBO NEXT QUERY GENERATION

Function 1
Best output so far: 7.99676981308551e-19
Best input so far:  [0.582812 0.421077]
Strategy:           Balanced search.
Feature importance: [0. 0.]
Local sensitivity:  [-0. -0.]
Next input:         [0.582812 0.421077]

Function 2
Best output so far: 0.5675786892822564
Best input so far:  [0.723607 0.276393]
Strategy:           Balanced search.
Feature importance: [0.4289 0.5711]
Local sensitivity:  [-0.0329 -0.0107]
Next input:         [0.718442 0.086744]

Function 3
Best output so far: -0.025522227618805376
Best input so far:  [0.147117 0.526235 0.849011]
Strategy:           Wider exploration because output is still negative.
Feature importance: [0.3802 0.1064 0.5134]
Local sensitivity:  [ 0.0029 -0.0017 -0.003 ]
Next input:         [0.155756 0.525144 0.938413]

Function 4
Best output so far: -4.5906554634204895
Best input so far:  [0.524137 0.403433 0.22003  0.482874]
Strategy:           Wider exploration because output is still negat